# Warm up
Let's start with some warm up exercises to get you familiar with the database and do some SQL querying.
Make sure to have downloaded the NBA dataset form Kaggle as described in the README.

In [ ]:
import sqlite3 as sql
import pandas as pd
from pathlib import Path

## TODO: Use pathlib to get the path to the data directory. Path(__file__) won't work in a Jupyter notebook.
DATA_PATH = Path.cwd() / 'data'
print(DATA_PATH)

## Create a SQLite connection and run some SQL queries

In [ ]:
con = sql.connect(DATA_PATH / 'nba.sqlite')  # connect to the database
query = 'SELECT * FROM game LIMIT 10'  # write a query that selects the first 10 rows from the game table
top_10_game = pd.read_sql(query, con)  # read the query into a pandas dataframe
top_10_game.head()  # print the dataframe

In [ ]:
# let's look at the schema of the database
# TODO: sqlite_master is the name of the table you want to inspeact: Replace TABLE_NAME with the name of the table you want to inspect
query = """
    SELECT *
    FROM sqlite_master
"""

pd.read_sql(query, con).sort_values('name')

In [ ]:
# let's see all the tables we have in the database
# TODO: Filter for tables only
query = """
    SELECT name
    FROM sqlite_master
    where type='table'
"""
pd.read_sql(query, con)

In [ ]:
# let's have a look at the columns in the game table
query = """
    PRAGMA table_info(game)
"""
pd.read_sql(query, con)

In [ ]:
# let's find the earliest recorded game
query = """
    SELECT game_date
    FROM game
    ORDER BY game_date
    LIMIT 1
"""

print(pd.read_sql(query, con))

# let's find the latest recorded game
query = """
    SELECT game_date
    FROM game
    ORDER BY game_date DESC
    LIMIT 1
"""

print(pd.read_sql(query, con))


In [ ]:
# TODO: Find four interesting facts about the data and write a query to find the answer
query = """
    SELECT wl_home, season_id, game_date,game_id, team_name_home, team_name_away, pts_home, pts_away
    FROM game
    WHERE season_type != 'All-Star'
"""
season_wl = pd.read_sql(query, con)
season_wl

In [ ]:
season_wl.dtypes
season_wl["season_id"] = season_wl["season_id"].astype(int)
season_wl["game_date"] = pd.to_datetime(season_wl["game_date"])

In [ ]:
season_wl.isna().sum()
season_wl[season_wl["wl_home"].isna()]
season_wl.loc[season_wl["game_id"] == "0010700072", "wl_home"] = "W"
season_wl.loc[season_wl["game_id"] == "0010800035", "wl_home"] = "L"

season_wl["game_id"].is_unique
season_wl[season_wl["game_id"].duplicated(keep = False)].sort_values("game_id")
season_wl = season_wl.drop_duplicates(subset=["game_id", "team_name_home", "team_name_away", "pts_home", "pts_away"])


In [ ]:
season_wl


In [ ]:
import matplotlib.pyplot as plt

# percentage of games won by the home team in each calendar year
season_wl["year"] = season_wl["game_date"].dt.year

home_win_pct = (
    season_wl.groupby("year")["wl_home"]
    .apply(lambda s: (s == "W").mean() * 100)
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(home_win_pct.index, home_win_pct.values, linewidth=2, marker="o", markersize=4)
ax.set_xlabel("Year")
ax.set_ylabel("Home team wins (%)")
ax.set_title("Percentage of games won by the home team, by year")
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
query = """
    SELECT person_id, season, round_number, round_pick, overall_pick, organization
    FROM draft_history
    WHERE overall_pick = 1
"""
draft_history = pd.read_sql(query, con)
draft_history 

In [ ]:
# load every pick (the cell above only kept overall_pick = 1)
query = """
    SELECT season, overall_pick, team_name, organization_type
    FROM draft_history
"""
draft_all = pd.read_sql(query, con)
draft_all["season"] = draft_all["season"].astype(int)
draft_all.head()

In [ ]:
# where draftees come from, by decade
draft_all["decade"] = draft_all["season"] // 10 * 10
draft_all["organization_type"] = draft_all["organization_type"].replace("", "Unknown")

origin = pd.crosstab(
    draft_all["decade"], draft_all["organization_type"], normalize="index"
) * 100

fig, ax = plt.subplots(figsize=(9, 4))
origin.plot(kind="bar", stacked=True, ax=ax)
ax.set_xlabel("Decade")
ax.set_ylabel("Share of draftees (%)")
ax.set_title("Where NBA draftees came from, by decade")
ax.legend(title="Organization type", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
# teams with the most top-5 picks
# some early drafts (1947-1965) have overall_pick = 0 (not recorded), so keep only picks 1-5
top5 = (
    draft_all[draft_all["overall_pick"].between(1, 5)]["team_name"]
    .value_counts()
    .head(10)
)

top5_sorted = top5.sort_values()
colors = plt.cm.tab10(range(len(top5_sorted)))  # one distinct colour per bar

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(top5_sorted.index, top5_sorted.values, color=colors)
ax.set_xlabel("Number of top-5 picks")
ax.set_ylabel("")
ax.set_title("Teams with the most top-5 draft picks, 1947-2023")
ax.grid(axis="x", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.show()